# Case Disruption Trend — the F/E/G decomposition by decision year

`case_disruption.parquet` gives one row per case. This aggregates it to one row per **decision
year**, which is the form the "is law becoming less disruptive?" question is asked in. Reads
this folder's own output only — no graph, no edge list, seconds to run.

## Output
`Case law/output/case_feg_disruption_trend.parquet`

`year`, `n`, and for each window `{3,5,10,all}` the mean of `CD`, `F`, `E`, `G`, `ni`, `nj`,
`nk`, plus `njfrac` = `nj / (ni + nj)` — the share of a case's citers that also cite its own
references, which is the consolidating half of CD read directly.

## Two things to read carefully

**Right truncation.** A case decided in 2018 cannot have a 10-year citation window. `CD_10` for
recent years is computed over a window that has not elapsed, so the late end of every
fixed-window series bends for a reason that has nothing to do with law. `CD_all` is worse, not
better: its window grows shorter the closer the case is to the snapshot. Read the series only
where the window has closed, and use `n` to see how much of a year survives.

**Undefined is not zero.** A case with no citers has `CD = NaN` and is excluded from that year's
mean, so `n` differs by window. Early years are thin on citers and late years are thin on
elapsed time, which pinches the series from both ends.

In [1]:
%%time
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Case law')
import cl_common as cl
OUT_FP = cl.out('case_feg_disruption_trend.parquet')
DIS, META = cl.out('case_disruption.parquet'), cl.out('case_metadata.parquet')
WIN = ['_3', '_5', '_10', '_all']
for f in (DIS, META):
    assert os.path.exists(f), f'{f} missing — run case_disruption / case_metadata first'

d = pd.read_parquet(DIS)
m = pd.read_parquet(META, columns=['case_id', 'decision_year'])
d = d.merge(m, on='case_id', how='inner')
d = d[d.decision_year.notna()]
d['decision_year'] = d.decision_year.astype(int)
# Every analysis starts at cl.YEAR_MIN (1800). The GRAPH is not filtered -- the CD of an
# 1805 case still counts its 1790 antecedents; only the reported population is floored.
_pre = int((d.decision_year < cl.YEAR_MIN).sum())
d = d[d.decision_year >= cl.YEAR_MIN]
print(f'floor {cl.YEAR_MIN}: dropped {_pre:,} pre-{cl.YEAR_MIN} cases from the trend')
print(f'{len(d):,} cases with a decision year')
for s in WIN:
    d[f'njfrac{s}'] = d[f'nj{s}'] / (d[f'ni{s}'] + d[f'nj{s}']).replace(0, np.nan)

floor 1800: dropped 1,795 pre-1800 cases from the trend
5,177,903 cases with a decision year


In [2]:
%%time
agg = {'n': ('CD_all', 'size')}
for s in WIN:
    for k in ('CD', 'F', 'E', 'G', 'ni', 'nj', 'nk', 'njfrac'):
        agg[f'{k}{s}_mean'] = (f'{k}{s}', 'mean')
tr = d.groupby('decision_year').agg(**agg).reset_index().rename(columns={'decision_year': 'year'})
# n per window: a case with no citers is NaN and must not count toward that window's mean
for s in WIN:
    tr[f'n{s}_defined'] = d.groupby('decision_year')[f'CD{s}'].count().values
tr = tr.sort_values('year').reset_index(drop=True)
tr.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(tr):,} rows, {len(tr.columns)} cols)')
print(f'  years {int(tr.year.min())}-{int(tr.year.max())}')
show = ['year', 'n', 'n_all_defined', 'CD_all_mean', 'CD_10_mean', 'CD_5_mean',
        'njfrac_all_mean', 'F_all_mean', 'E_all_mean', 'G_all_mean']
display(tr[show].tail(20).round(4))

WROTE /project/jevans/Dawoon/Science of Science/Case law/output/case_feg_disruption_trend.parquet  (221 rows, 38 cols)
  years 1800-2020


,year,n,n_all_defined,CD_all_mean,CD_10_mean,CD_5_mean,njfrac_all_mean,F_all_mean,E_all_mean,G_all_mean
201,2001,67935,42467,0.0233,0.0204,0.0164,0.6125,0.1235,0.5627,0.3138
202,2002,67610,42442,0.0233,0.0206,0.0162,0.6148,0.1212,0.5661,0.3128
203,2003,68927,41406,0.0249,0.0227,0.0180,0.6137,0.1160,0.5674,0.3166
204,2004,68869,40345,0.0254,0.0234,0.0191,0.6134,0.1119,0.5689,0.3191
205,2005,72889,41627,0.0252,0.0237,0.0191,0.6264,0.1056,0.5834,0.3111
206,2006,76381,40943,0.0226,0.0214,0.0174,0.6264,0.1032,0.5841,0.3126
207,2007,74911,40725,0.0190,0.0185,0.0152,0.6335,0.0935,0.5951,0.3114
208,2008,72974,39389,0.0185,0.0183,0.0155,0.6419,0.0879,0.6060,0.3062
209,2009,73493,38297,0.0178,0.0178,0.0151,0.6567,0.0811,0.6233,0.2957
210,2010,72636,36622,0.0162,0.0162,0.0138,0.6601,0.0751,0.6297,0.2951


## Plot

Small multiples, one panel per window, `CD` mean by decision year. The vertical guide marks the
last year whose window has fully elapsed given the 2020 snapshot — everything to its right is
truncated, and is drawn faint rather than hidden so the truncation is visible instead of
cropped away.

In [3]:
import matplotlib.pyplot as plt
SNAP = int(tr.year.max())
BLUE, ORANGE, INK, MUTED, GRID, SURFACE = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e', '#e6e5e1', '#fcfcfb'
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, layout='constrained')
for ax, s in zip(axes.ravel(), WIN):
    # An all-time window never closes -- it shortens toward the snapshot -- so it takes the
    # widest fixed window's horizon as a conservative stand-in rather than no guard at all.
    w = {'_3': 3, '_5': 5, '_10': 10, '_all': 10}[s]
    closed = SNAP - w
    g = tr[tr[f'n{s}_defined'] >= 30]
    solid = g[g.year <= closed]; faint = g[g.year >= closed]
    ax.plot(solid.year, solid[f'CD{s}_mean'], lw=2, color=BLUE, solid_capstyle='round')
    ax.plot(faint.year, faint[f'CD{s}_mean'], lw=2, color=BLUE, alpha=.30)
    ax.axvline(closed, color=MUTED, lw=1, ls='--')
    ax.annotate(('window closes ' if s != '_all' else '10y stand-in ') + f'{closed}',
                xy=(closed, ax.get_ylim()[1]), xytext=(-6, -12),
                textcoords='offset points', ha='right', va='top', fontsize=8, color=MUTED)
    ax.axhline(0, color=MUTED, lw=.8, ls=':')
    ax.set_title(f"mean CD{s}   (cases with >=30 defined)", color=INK, loc='left', pad=8)
    ax.set_ylabel('CD', color=MUTED); ax.set_facecolor(SURFACE)
    ax.grid(axis='y', color=GRID, lw=.8, zorder=0); ax.set_axisbelow(True)
    ax.tick_params(colors=MUTED, length=3)
    for sp in ax.spines.values():
        sp.set_color(GRID)
for ax in axes[-1]:
    ax.set_xlabel('decision year', color=MUTED)
fig.suptitle('Case law disruption by decision year — faint = citation window not yet elapsed',
             fontsize=12, color=INK)
fig.patch.set_facecolor(SURFACE)
os.makedirs(f'{cl.BASE}/Figures', exist_ok=True)
fig.savefig(f'{cl.BASE}/Figures/case_disruption_trend.png', dpi=200, bbox_inches='tight',
            facecolor=SURFACE)
plt.show()
print(f'-> {cl.BASE}/Figures/case_disruption_trend.png')

-> /project/jevans/Dawoon/Science of Science/Case law/Figures/case_disruption_trend.png
